In [ ]:
from pathlib import Path
import os, sys, re, json, hashlib, random, warnings, time, zipfile
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, cv2
from PIL import Image
from dataclasses import dataclass,asdict
from scipy.ndimage import label as ndlabel, binary_dilation, distance_transform_edt
connected_label = ndlabel
from skimage.morphology import skeletonize
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor,RandomForestRegressor
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, mean_absolute_error
from sklearn.model_selection import GroupKFold
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
from collections import Counter
@dataclass
class CFG:
 root:str='/kaggle/working/VASC_Guard_RiskBenefit_Stage5_Structural_Validated'
 artifact_hint:str='/kaggle/input/datasets/jitushimul/vasguard/VASC_Guard_RUN_001'
 drive_root:str='/kaggle/working/DRIVE'
 chase_root:str='/kaggle/input/datasets/khoongweihao/chasedb1'
 stare_root:str='/kaggle/input/datasets/jitushimul/stare-data/STARE'
 drive_repo:str='Zomba/DRIVE-digital-retinal-images-for-vessel-extraction'
 validation_ids:tuple=('24','28','34','38')
 seeds:tuple=(42,123,2026)
 models:tuple=('selective_guard',)
 base_channels:int=24
 patch:int=384
 repair_weight:float=.25
 allow_download_drive:bool=True
 max_candidates_per_image:int=100
 max_bridge_distance:int=14
 min_component_area:int=3
 max_component_area:int=90
 budgets:tuple=(.05,.10,.20,.40,.60,.80,1.00)
 bootstraps:int=2000
 selected_estimator:str='ridge'
 utility_harm_lambda:float=1.0
 structural_harm_lambda:float=0.5
 structural_bad_threshold:float=0.50
 random_seed:int=42
 save_maps:bool=False
cfg=CFG(); OUT=Path(cfg.root)
# NumPy 1.x in some Kaggle images lacks trapezoid.
if not hasattr(np, 'trapezoid'): np.trapezoid = np.trapz
for d in ('audit','candidate_tables','metrics','figures','maps','models','reports'): (OUT/d).mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(42);np.random.seed(42);torch.manual_seed(42)
print('DEVICE',DEVICE,'OUTPUT',OUT)


In [ ]:
# ORIGINAL v9 MODEL DEFINITIONS — copied without architecture modifications
# 6. Model building blocks. All models share comparable base width.
class ConvBlock(nn.Module):
 def __init__(self,a,b):
    super().__init__();self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(True),nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(True))
 def forward(self,x):return self.net(x)
class AttGate(nn.Module):
 def __init__(self,g,x):
    super().__init__();k=max(x//2,8);self.a=nn.Conv2d(g,k,1);self.b=nn.Conv2d(x,k,1);self.c=nn.Conv2d(k,1,1)
 def forward(self,skip,up):return skip*torch.sigmoid(self.c(F.relu(self.a(up)+self.b(skip))))
class Up(nn.Module):
 def __init__(self,ain,skip,out,attention=False):
    super().__init__();self.g=AttGate(ain,skip) if attention else None;self.conv=ConvBlock(ain+skip,out)
 def forward(self,x,s):
    x=F.interpolate(x,size=s.shape[-2:],mode='bilinear',align_corners=False)
    if self.g is not None:s=self.g(s,x)
    return self.conv(torch.cat([x,s],dim=1))
class UNet(nn.Module):
 def __init__(self,c=24,attention=False):
    super().__init__();self.enc0=ConvBlock(3,c);self.enc1=ConvBlock(c,c*2);self.enc2=ConvBlock(c*2,c*4);self.enc3=ConvBlock(c*4,c*8)
    self.pool=nn.MaxPool2d(2);self.up2=Up(c*8,c*4,c*4,attention);self.up1=Up(c*4,c*2,c*2,attention);self.up0=Up(c*2,c,c,attention)
    self.out=nn.Conv2d(c,1,1);self.feature_channels=c
 def features(self,x):
    a=self.enc0(x);b=self.enc1(self.pool(a));c=self.enc2(self.pool(b));d=self.enc3(self.pool(c))
    return self.up0(self.up1(self.up2(d,c),b),a)
 def forward(self,x):return {'logits':self.out(self.features(x))}
class UNetPP(nn.Module):
 def __init__(self,c=24):
    super().__init__();self.enc0=ConvBlock(3,c);self.enc1=ConvBlock(c,2*c);self.enc2=ConvBlock(2*c,4*c);self.enc3=ConvBlock(4*c,8*c)
    self.x01=ConvBlock(c+2*c,c);self.x11=ConvBlock(2*c+4*c,2*c);self.x21=ConvBlock(4*c+8*c,4*c)
    self.x02=ConvBlock(2*c+2*c,c);self.x12=ConvBlock(4*c+4*c,2*c);self.x03=ConvBlock(3*c+2*c,c);self.out=nn.Conv2d(c,1,1)
 def forward(self,x):
    p=F.max_pool2d;up=lambda z,like:F.interpolate(z,size=like.shape[-2:],mode='bilinear',align_corners=False)
    a=self.enc0(x);b=self.enc1(p(a,2));c=self.enc2(p(b,2));d=self.enc3(p(c,2))
    a1=self.x01(torch.cat([a,up(b,a)],1));b1=self.x11(torch.cat([b,up(c,b)],1));c1=self.x21(torch.cat([c,up(d,c)],1))
    a2=self.x02(torch.cat([a,a1,up(b1,a)],1));b2=self.x12(torch.cat([b,b1,up(c1,b)],1))
    a3=self.x03(torch.cat([a,a1,a2,up(b2,a)],1));return {'logits':self.out(a3)}

class VascGuard(nn.Module):
 def __init__(self,c=24,uncertainty=True,repair=True):
    super().__init__();self.backbone=UNet(c)
    self.uncertainty=uncertainty;self.repair=repair
    if uncertainty:self.error_head=nn.Sequential(nn.Conv2d(c,c,3,padding=1),nn.ReLU(),nn.Conv2d(c,1,1))
    if repair:
        self.repair_head=nn.Sequential(ConvBlock(c+3,c),nn.Conv2d(c,1,1))
 def forward(self,x):
    feat=self.backbone.features(x);base=self.backbone.out(feat);out={'base_logits':base}
    prob=torch.sigmoid(base)
    error_logits=self.error_head(feat) if self.uncertainty else None
    err=torch.sigmoid(error_logits) if self.uncertainty else torch.ones_like(prob)
    if self.uncertainty:out['error_logits']=error_logits
    # Dilated local candidate connections; learned residual corrected only where risk/evidence support it.
    if self.repair:
        neighbor=F.max_pool2d(prob,5,stride=1,padding=2)
        residual=self.repair_head(torch.cat([feat,prob,neighbor,err],1))
        logits=base+cfg.repair_weight*err*residual
    else:logits=base
    out['logits']=logits
    return out


class SelectiveVascGuard(nn.Module):
    """Hypothesis model: learned connection proposals with independently gated residual changes.

    Gate is a calibrated *decision score*, NOT an epistemic uncertainty estimate.
    A local 5x5 neighborhood does not guarantee valid vascular connectivity.
    """
    def __init__(self,c=24,use_gate=True,use_connectivity=True):
        super().__init__()
        self.backbone=UNet(c)
        self.use_gate=use_gate
        self.use_connectivity=use_connectivity
        self.error_head=nn.Sequential(ConvBlock(c+1,c),nn.Conv2d(c,1,1))
        self.proposal_head=nn.Sequential(ConvBlock(c+4,c),nn.Conv2d(c,1,1))
        self.gate_head=nn.Sequential(ConvBlock(c+4,c),nn.Conv2d(c,1,1))
    def forward(self,x):
        feat=self.backbone.features(x)
        base=self.backbone.out(feat)
        p=torch.sigmoid(base)
        neighbor=F.max_pool2d(p,5,stride=1,padding=2)
        gap=(neighbor-p).clamp(0,1)
        error_logits=self.error_head(torch.cat([feat,p],dim=1))
        risk=torch.sigmoid(error_logits)
        evidence=gap if self.use_connectivity else torch.ones_like(gap)
        evidence_input=torch.cat([feat,p,neighbor,risk,evidence],dim=1)
        proposal=self.proposal_head(evidence_input)
        gate_logits=self.gate_head(evidence_input)
        gate=torch.sigmoid(gate_logits) if self.use_gate else torch.ones_like(p)
        # Bounded residual: prevents unbounded repair changes, but not incorrect bridges.
        correction=cfg.repair_weight*torch.tanh(proposal)*gate*evidence
        logits=base+correction
        return dict(logits=logits,base_logits=base,error_logits=error_logits,
                    gate_logits=gate_logits,gate=gate,proposal=proposal,
                    correction=correction,connection_evidence=evidence)

def create_model(name):
    c=cfg.base_channels
    if name in ('unet','cldice_unet'):return UNet(c)
    if name=='attention_unet':return UNet(c,True)
    if name=='unetpp':return UNetPP(c)
    if name=='guard_no_uncertainty':return VascGuard(c,False,True)
    if name=='guard_no_repair':return VascGuard(c,True,False)
    if name=='vasc_guard':return VascGuard(c,True,True)
    if name=='selective_guard':return SelectiveVascGuard(c,True,True)
    if name=='selective_no_gate':return SelectiveVascGuard(c,False,True)
    if name=='selective_no_connectivity':return SelectiveVascGuard(c,True,False)
    raise KeyError(name)

for name in cfg.models:
    m=create_model(name)
    with torch.no_grad():
        o=m(torch.zeros(1,3,64,64))
    assert o['logits'].shape==(1,1,64,64),name
    print(name,'parameters:',sum(p.numel() for p in m.parameters()))

In [ ]:
# ORIGINAL v9 FULL-RESOLUTION INFERENCE AND METRICS
# 8. Full-resolution sliding-window inference; never resize test labels or test images.
@torch.no_grad()
def predict_maps(model,rgb,patch=None):
    model.eval();patch=patch or cfg.patch
    h,w=rgb.shape[:2];stride=max(64,patch//2)
    hh=max(h,patch);ww=max(w,patch)
    img=cv2.copyMakeBorder(rgb,0,hh-h,0,ww-w,cv2.BORDER_REFLECT_101).astype(np.float32)/255.
    h2,w2=img.shape[:2]
    ys=list(range(0,h2-patch+1,stride));xs=list(range(0,w2-patch+1,stride))
    if ys[-1]!=h2-patch:ys.append(h2-patch)
    if xs[-1]!=w2-patch:xs.append(w2-patch)
    sums={};cnt=np.zeros((h2,w2),np.float32)
    window=np.outer(np.hanning(patch)+.05,np.hanning(patch)+.05).astype(np.float32)
    for y in ys:
        for x in xs:
            chunk=np.ascontiguousarray(img[y:y+patch,x:x+patch].transpose(2,0,1))
            out=model(torch.from_numpy(chunk)[None].to(DEVICE))
            maps={'final':torch.sigmoid(out['logits'])}
            if 'base_logits' in out:maps['base']=torch.sigmoid(out['base_logits'])
            if 'error_logits' in out:maps['risk']=torch.sigmoid(out['error_logits'])
            if 'gate' in out:maps['gate']=out['gate']
            for name,t in maps.items():
                if name not in sums:sums[name]=np.zeros((h2,w2),np.float32)
                val=t[0,0].detach().float().cpu().numpy()
                sums[name][y:y+patch,x:x+patch]+=window*val
            cnt[y:y+patch,x:x+patch]+=window
    return {k:np.clip(v[:h,:w]/np.maximum(cnt[:h,:w],1e-8),0,1) for k,v in sums.items()}

def predict_prob(model,rgb,patch=None):
    return predict_maps(model,rgb,patch)['final']

def safe_metrics(gt,probs,thr=.5,fov=None):
    gt=gt.astype(bool);pred=probs>=thr
    if fov is None:fov=np.ones_like(gt,bool)
    gt=gt&fov;pred=pred&fov
    tp=np.count_nonzero(gt&pred);fp=np.count_nonzero(~gt&pred&fov);fn=np.count_nonzero(gt&~pred&fov);tn=np.count_nonzero(~gt&~pred&fov)
    eps=1e-8
    dice=2*tp/(2*tp+fp+fn+eps)
    iou=tp/(tp+fp+fn+eps)
    prec=tp/(tp+fp+eps);rec=tp/(tp+fn+eps)
    spec=tn/(tn+fp+eps)
    s1=skeletonize(pred);s2=skeletonize(gt)
    tprec=(s1&gt).sum()/(s1.sum()+eps);tsens=(s2&pred).sum()/(s2.sum()+eps)
    cl=2*tprec*tsens/(tprec+tsens+eps)
    n_pred=connected_label(pred,structure=np.ones((3,3)))[1];n_gt=connected_label(gt,structure=np.ones((3,3)))[1]
    return dict(dice=dice,iou=iou,precision=prec,sensitivity=rec,specificity=spec,cldice=cl,
                component_abs_error=abs(n_pred-n_gt),pred_components=n_pred,gt_components=n_gt)

def ece_score(gt,prob,fov,bins=12):
    g=gt[fov].astype(float);p=prob[fov];score=0.;n=max(1,len(p))
    for lo,hi in zip(np.linspace(0,1,bins+1)[:-1],np.linspace(0,1,bins+1)[1:]):
        q=(p>=lo)&(p<hi if hi<1 else p<=hi)
        if q.any():score+=q.sum()/n*abs(g[q].mean()-p[q].mean())
    return float(score)



In [ ]:
# Load RGB/GT masks and field of view: compare in native pixels.
def open_rgb(p):return np.asarray(Image.open(p).convert('RGB'))
def open_mask(p):return (np.asarray(Image.open(p).convert('L'))>127)
def get_fov(rgb):
 mx=np.max(rgb,axis=2)
 raw=(mx>22).astype(np.uint8)
 n,cc,stats,cents=cv2.connectedComponentsWithStats(raw,8)
 if n>1:
  k=1+np.argmax(stats[1:,cv2.CC_STAT_AREA]);fov=(cc==k).astype(np.uint8)
 else:fov=np.ones(raw.shape,np.uint8)
 fov=cv2.morphologyEx(fov,cv2.MORPH_CLOSE,np.ones((13,13),np.uint8))
 return fov.astype(bool)


In [ ]:
# Locate old artifacts; inspect checkpoint structure before loading.
roots=[Path(cfg.artifact_hint),Path('/kaggle/input')]
allck=[]
for r in roots:
 if r.exists():allck += list(r.rglob('*_best.pt'))
allck=sorted(set(allck))
print('Found checkpoints:',len(allck));print(*[str(x) for x in allck[:35]],sep='\n')
expected={(name,seed) for name in cfg.models for seed in cfg.seeds}
def find_checkpoint(name,seed):
 matches=[p for p in allck if p.name==f'{name}_seed{seed}_best.pt']
 if len(matches)!=1:raise RuntimeError(f'Expected exactly one checkpoint for {name} seed {seed}, found {matches}')
 return matches[0]
registry={}
for name,seed in sorted(expected):
 path=find_checkpoint(name,seed)
 state=torch.load(path,map_location='cpu',weights_only=False)
 assert isinstance(state,dict) and isinstance(state.get('model'),dict),'Unexpected checkpoint format: '+str(path)
 if state.get('name',name)!=name or int(state.get('seed',seed))!=seed:raise RuntimeError('Checkpoint identity mismatch: '+str(path))
 model=create_model(name)
 model.load_state_dict(state['model'],strict=True)
 model.eval().to(DEVICE)
 registry[name,seed]=(model,path)
 print('STRICT LOAD OK',name,seed,'epoch',state.get('epoch'),'n tensors',len(state['model']))
pd.DataFrame([{'model':k[0],'seed':k[1],'checkpoint':str(v[1])} for k,v in registry.items()]).to_csv(OUT/'audit/checkpoints.csv',index=False)


In [ ]:
# DATASET ACQUISITION — EXACT SOURCES AND FILENAMES FROM ORIGINAL v9 NOTEBOOK
# DRIVE: Zomba/DRIVE-digital-retinal-images-for-vessel-extraction (Hugging Face)
# CHASE_DB1: /kaggle/input/datasets/khoongweihao/chasedb1
# STARE: /kaggle/input/datasets/jitushimul/stare-data/STARE
# The old saved absolute /kaggle/working/DRIVE/... paths are not persistent across sessions.
import importlib.util, subprocess
if importlib.util.find_spec('huggingface_hub') is None:
 subprocess.check_call([sys.executable,'-m','pip','install','-q','huggingface_hub'])
from huggingface_hub import snapshot_download

def verify_original_drive_layout(root):
 root=Path(root)
 required=[]
 for i in range(21,41):
  required.extend([root/'train'/'input'/f'{i}.tif',root/'train'/'label'/f'{i}.png'])
 for i in range(1,21):
  required.extend([root/'val'/'input'/f'{i:02d}.tif',root/'val'/'label'/f'{i:02d}_manual1.png'])
 return [p for p in required if not p.is_file()]

missing_drive=verify_original_drive_layout(cfg.drive_root)
if missing_drive:
 print('Original v9 DRIVE mirror missing',len(missing_drive),'files. Fetching original dataset repository.')
 if not cfg.allow_download_drive:
  raise FileNotFoundError('Enable cfg.allow_download_drive=True and Kaggle Internet, or attach the exact v9 DRIVE mirror. Missing: '+str(missing_drive[:6]))
 try:
  snapshot_download(repo_id=cfg.drive_repo,repo_type='dataset',local_dir=cfg.drive_root)
 except Exception as exc:
  raise RuntimeError('Could not fetch the exact v9 DRIVE repository. Turn on Kaggle Internet (Settings) or attach a local copy of the same repository. Original error: '+repr(exc)) from exc
missing_drive=verify_original_drive_layout(cfg.drive_root)
if missing_drive:
 raise FileNotFoundError('DRIVE mirror is incomplete. Missing examples: '+str(missing_drive[:8]))
print('VERIFIED: All 40 original v9 DRIVE image/primary annotation pairs are present.')

def check_external_v9_roots():
 for name,root in [('CHASE_DB1',cfg.chase_root),('STARE',cfg.stare_root)]:
  if not Path(root).is_dir():
   raise FileNotFoundError(f'{name} dataset is missing at {root}. Attach the ORIGINAL v9 Kaggle dataset at this root. Available datasets: '+str([str(p) for p in Path('/kaggle/input').iterdir()]) if Path('/kaggle/input').exists() else f'{name} dataset missing; /kaggle/input is absent')
 check_ch=list(Path(cfg.chase_root).rglob('Image_*'))
 check_st=list(Path(cfg.stare_root).rglob('images'))
 print('CHASE source exists:',cfg.chase_root,'| STARE source exists:',cfg.stare_root)
check_external_v9_roots()


In [ ]:
# EXACT v9 dataset mapping. Do not resolve by basename alone: masks may share image names.
manifest_files=[]
for r in roots:
 if r.exists():manifest_files.extend(r.rglob('locked_manifest.csv'))
if not manifest_files:raise FileNotFoundError('Original locked_manifest.csv absent. Attach original VASC_Guard_RUN_001 artifact dataset.')
manifest_files=sorted(set(manifest_files),key=lambda p:(len(p.parts),str(p)))
if len(manifest_files)>1:print('Available manifests:',[str(p) for p in manifest_files])
original=pd.read_csv(manifest_files[0]).fillna('')
assert len(original)==88 and {'dataset','id','role','image','mask'}.issubset(original.columns)

def resolve_v9(path,dataset,field):
 """Resolve saved paths only against the exact original dataset layout and ID."""
 p=Path(str(path))
 if p.is_file():return str(p)
 if dataset=='DRIVE':
  # Original source: /kaggle/working/DRIVE/{train,val}/{input,label}/{filename}
  parts=list(p.parts)
  branch=next((x for x in ('train','val') if x in parts),None)
  if branch is None:raise FileNotFoundError('Cannot identify DRIVE split from locked path: '+str(path))
  source=Path(cfg.drive_root)/branch/('input' if field=='image' else 'label')/p.name
  if not source.is_file():raise FileNotFoundError(f'Missing verified v9 DRIVE {field}: {source}')
  return str(source)
 if dataset=='STARE':
  root=Path(cfg.stare_root)
  sub={'image':'images','mask':'masks_ah','secondary':'masks_vk'}[field]
  matches=list(root.rglob(sub+'/'+p.name))
 elif dataset=='CHASE_DB1':
  root=Path(cfg.chase_root)
  if field=='image':
   matches=[q for q in root.rglob(p.name) if q.stem.lower().startswith('image_')]
  else:
   suffix='_1stHO.png' if field=='mask' else '_2ndHO.png'
   basename=str(p.name)
   if not basename.lower().endswith(suffix.lower()):
    # Preserve the original image identifier, NOT positional matching.
    stem=str(p.name).split('_')[0] if False else None
    raise FileNotFoundError('Unexpected CHASE observer filename in original manifest: '+str(path))
   matches=[q for q in root.rglob(p.name) if q.name.lower()==p.name.lower()]
 else:raise KeyError(dataset)
 matches=sorted(set(matches))
 if len(matches)!=1:
  raise FileNotFoundError(f'{dataset} {field} expected one original v9 match for {path}; got {[str(q) for q in matches[:8]]}')
 return str(matches[0])

manifest=original.copy()
for idx,r in manifest.iterrows():
 for field in ('image','mask','secondary'):
  if field=='secondary' and not str(r.get(field,'')).strip():continue
  manifest.at[idx,field]=resolve_v9(r[field],r.dataset,field)
for r in manifest.itertuples():
 with Image.open(r.image) as im, Image.open(r.mask) as gt:
  assert im.size==gt.size,(r.dataset,r.id,'shape mismatch')
  fg=(np.asarray(gt.convert('L'))>127).mean()
  assert 0.001<fg<0.30,(r.dataset,r.id,'annotation QC',fg,r.mask)
 if getattr(r,'secondary',''):
  with Image.open(r.secondary) as secondary:assert secondary.size==im.size,(r.dataset,r.id,'secondary mismatch')
assert (manifest.role=='train').sum()==16 and (manifest.role=='validation').sum()==4
assert (manifest.role=='test').sum()==20 and (manifest.role=='external_test').sum()==48
assert not set(manifest[manifest.role=='train'].image)&set(manifest[manifest.role!='train'].image)
manifest.to_csv(OUT/'audit/resolved_manifest.csv',index=False)
print('VERIFIED ORIGINAL v9 DATASET PAIRS:')
print(manifest.groupby(['dataset','role']).size().to_string())
print(manifest[['dataset','id','role','image','mask']].groupby('dataset').head(2).to_string(index=False))
# Same v9 validation-only checkpoint thresholds, never external outcomes.
sel_files=[]
for root in roots:
 if root.exists():sel_files+=list(root.rglob('model_selection_VALIDATION_ONLY.csv'))
if not sel_files:raise FileNotFoundError('Original validation-only threshold table missing.')
selection=pd.read_csv(sorted(set(sel_files))[0])
assert {'name','seed','validation_threshold'}.issubset(selection.columns)
thresholds={}
for (name,seed),_ in registry.items():
 rows=selection[(selection.name==name)&(selection.seed.astype(int)==seed)]
 assert len(rows)==1,(name,seed)
 thresholds[name,seed]=float(rows.iloc[0].validation_threshold)
print('Frozen validation thresholds:',thresholds)


In [ ]:
# Unit checks, identifiers, and a fixed protocol ledger.
protocol={'starting_prediction':'v9 base_logits output (pre-learned-residual) from selective_guard',
'candidate_generator':'Stage3 deterministic morphological endpoint bridges + small isolated component removals; separate from v9 learned residual repair',
'candidate_labels':'train/validation ground truth only for estimator selection; evaluation GT for reporting only',
'evaluation_status':'exploratory, external results previously inspected',
'cost':'number of changed in-FOV pixels per candidate, non-overlapping pool',
'estimator':'expected pixel-correction yield and independent structural-harm proxy (not complementary), fitted only DRIVE train; CV groups by image; v9 training-image outputs are in-sample',
'features':'ground-truth-free candidate morphological/probability/risk/gate features',
'heldout':'DRIVE original test, STARE and CHASE_DB1; no tuning',
'budgets':list(cfg.budgets),'checkpoint_names':list(cfg.models),'seeds':list(cfg.seeds)}
(OUT/'audit/protocol.json').write_text(json.dumps(protocol,indent=2))
assert abs((2*1/(2*1+0+0))-1)<1e-9
print(json.dumps(protocol,indent=2))


In [ ]:
# Candidate generator — deterministic and independent of ground truth.
# Types: endpoint bridge additions and isolated small-component removals.
def endpoint_pixels(mask):
 sk=skeletonize(mask)
 nbr=cv2.filter2D(sk.astype('uint8'),-1,np.ones((3,3),np.uint8),borderType=cv2.BORDER_CONSTANT)
 return np.argwhere(sk&(nbr==2))

def candidates_for(base_prob,risk,gate,thr,fov):
 base=(base_prob>=thr)&fov
 H,W=base.shape; component,ncomp=ndlabel(base,structure=np.ones((3,3),int))
 eps=endpoint_pixels(base); opts=[]
 if len(eps)>1800:eps=eps[np.linspace(0,len(eps)-1,1800).astype(int)]
 # Each candidate is a map of local integer pixel indices, requested binary state.
 if len(eps)>1:
  for ia,pt in enumerate(eps):
   ds=np.sum((eps-pt)**2,axis=1)
   for ib in np.where((ds>=4)&(ds<=cfg.max_bridge_distance**2))[0]:
    if ib<=ia:continue
    a,b=pt,eps[ib]
    if component[tuple(a)]==component[tuple(b)]:continue
    line=np.zeros((H,W),np.uint8)
    cv2.line(line,(int(a[1]),int(a[0])),(int(b[1]),int(b[0])),1,1)
    changed=np.flatnonzero((line>0)&(~base)&fov)
    if not 2<=len(changed)<=cfg.max_bridge_distance+2:continue
    # Rank candidates by preliminary no-GT evidence, cap runtime.
    score=float(base_prob.ravel()[changed].mean())
    opts.append(('bridge',changed,1,score,float(np.sqrt(ds[ib]))))
 for k in range(1,ncomp+1):
  idx=np.flatnonzero(component==k)
  if cfg.min_component_area<=len(idx)<=cfg.max_component_area:
   score=float(1-base_prob.ravel()[idx].mean())
   opts.append(('remove',idx,0,score,float(len(idx))))
 opts.sort(key=lambda x:(-x[3],x[0],len(x[1]),int(x[1][0])))
 chosen=[];occupied=np.zeros(H*W,bool)
 for kind,idx,target,score,geom in opts:
  if occupied[idx].any():continue
  occupied[idx]=True
  chosen.append({'type':kind,'idx':idx,'target':target,'heuristic':score,'geometry':geom,'cost':len(idx)})
  if len(chosen)>=cfg.max_candidates_per_image:break
 for j,c in enumerate(chosen):c['local_id']=j
 return base,chosen

def extract_features(c,p,risk,gate,base):
 idx=c['idx']; v=p.ravel()[idx]; u=risk.ravel()[idx];g=gate.ravel()[idx]
 x={'cost':len(idx),'log_cost':np.log1p(len(idx)),'kind_bridge':int(c['type']=='bridge'),
    'prob_mean':float(v.mean()),'prob_std':float(v.std()),'prob_min':float(v.min()),
    'risk_mean':float(u.mean()),'risk_max':float(u.max()),'gate_mean':float(g.mean()),
    'gate_std':float(g.std()),'geometry':c['geometry'],
    'heuristic':c['heuristic']}
 return x

def score_overlap(pred,gt):
 tp=np.sum(pred&gt);fp=np.sum(pred&~gt);fn=np.sum(~pred&gt)
 dice=2*tp/max(1,2*tp+fp+fn)
 iou=tp/max(1,tp+fp+fn)
 sp=skeletonize(pred);sg=skeletonize(gt)
 precision=(sp&gt).sum()/max(1,sp.sum()); recall=(sg&pred).sum()/max(1,sg.sum())
 cl=2*precision*recall/max(1e-9,precision+recall)
 return float(dice),float(iou),float(cl)

def observed_outcome(c,base,gt):
 changed=c['idx'];pre=base.ravel()[changed];post=np.full(len(changed),bool(c['target']))
 corrected=int(np.sum((pre!=gt.ravel()[changed])&(post==gt.ravel()[changed])))
 damaged=int(np.sum((pre==gt.ravel()[changed])&(post!=gt.ravel()[changed])))
 assert corrected+damaged==len(changed)
 return corrected,damaged
def structural_outcome(c,base,gt):
 """Ground-truth-derived structural harm supervision (development ONLY).
 Bridge: false supported-connection proxy = fraction of added bridge pixels not in GT;
 removal: fraction of removed pixels on GT skeleton; not proof of anatomical bridges.
 Returns risk proxy in [0,1] and binary event. Neither is algebraic complement of benefit.
 """
 idx=c['idx']; target=c['target']; g=gt.ravel()[idx]
 if c['type']=='bridge':
  proxy=float(np.mean(~g))
 else:
  # Removal of true skeleton may break a GT vessel path.
  sk=skeletonize(gt)
  proxy=float(np.mean(sk.ravel()[idx]))
 return proxy,int(proxy>cfg.structural_bad_threshold)

# Unit tests
b=np.zeros((10,10),bool); g=b.copy();idx=np.array([12,15,19]);g.ravel()[idx[:2]]=1
c={'idx':idx,'target':1};assert observed_outcome(c,b,g)==(2,1)
assert structural_outcome({'type':'bridge','idx':idx,'target':1},b,g)[0]==1/3
print('Candidate, structural-label and correction tests PASS')


In [ ]:
# Frozen checkpoint inference and candidate extraction; GT is never passed to generator.
allrecords=[];cache={};allcount=[]
for (name,seed),(model,path) in registry.items():
 for r in manifest.itertuples():
  rgb=open_rgb(r.image);gt=open_mask(r.mask).astype(bool); fov=get_fov(rgb)
  maps=predict_maps(model,rgb)
  if 'base' not in maps:raise RuntimeError('Checkpoint lacks base logits: '+str(path))
  p=maps['base'];risk=maps.get('risk',4*p*(1-p));gate=maps.get('gate',np.ones_like(p))
  base,candidates=candidates_for(p,risk,gate,thresholds[name,seed],fov)
  cachekey=(name,seed,r.dataset,str(r.id));cache[cachekey]=(base,gt,fov,p,risk,gate,candidates)
  for c in candidates:
   features=extract_features(c,p,risk,gate,base)
   rec={'model':name,'seed':seed,'dataset':r.dataset,'id':str(r.id),'role':r.role,
        'group':str(r.group),'candidate_id':f'{name}_{seed}_{r.dataset}_{r.id}_{c["local_id"]}',
        'local_id':c['local_id'],'candidate_type':c['type'],**features}
   # Outcome labels are only used to build DRIVE development targets or to evaluate policies later.
   if r.role in ('train','validation'):
    correct,harm=observed_outcome(c,base,gt)
    structural_proxy,structural_event=structural_outcome(c,base,gt)
    rec.update(corrected=correct,damaged=harm,net=correct-harm,utility_per_pixel=(correct-harm)/c['cost'],beneficial=int(correct>harm),structural_proxy=structural_proxy,structural_event=structural_event)
   allrecords.append(rec)
  allcount.append({'model':name,'seed':seed,'dataset':r.dataset,'role':r.role,'id':str(r.id),'n_candidates':len(candidates),'cost_pool':sum(x['cost'] for x in candidates)})
  print('Cached',name,seed,r.dataset,r.id,len(candidates),flush=True)
records=pd.DataFrame(allrecords);counts=pd.DataFrame(allcount)
records.to_csv(OUT/'candidate_tables/candidates_without_external_labels.csv',index=False)
counts.to_csv(OUT/'audit/candidate_availability.csv',index=False)
print(counts.groupby(['dataset','role'])[['n_candidates','cost_pool']].agg(['mean','min','max']))
if records.empty or not (records.role=='train').any():raise RuntimeError('No DRIVE training candidate examples; cannot learn benefit estimator.')


In [ ]:
# Stage 5: correction yield (count), damage yield (count), and non-complementary
# structural harm target (bridge unsupported fraction / true-vessel skeleton removal).
# Candidate-level model development uses DRIVE TRAIN only, grouped by unique image IDs.
feature_cols=['cost','log_cost','kind_bridge','prob_mean','prob_std','prob_min',
              'risk_mean','risk_max','gate_mean','gate_std','geometry','heuristic']
dev=records[(records.dataset=='DRIVE')&(records.role=='train')].copy()
val=records[(records.dataset=='DRIVE')&(records.role=='validation')].copy()
assert len(dev)>8 and dev.id.nunique()>=4
assert set(dev.id.astype(str)).isdisjoint(set(val.id.astype(str)))
# Fractions are natural within-change units. The *structural target* is independent
# of the correction/damage identity and has different consequences for intervention.
for frame in (dev,val):
 frame['correct_yield']=frame.corrected/frame.cost
 frame['damage_yield']=frame.damaged/frame.cost
 assert np.allclose(frame.correct_yield+frame.damage_yield,1.0)
 assert frame.structural_proxy.between(0,1).all()

def fit_regression(train,field):
 estimator=Pipeline([('imputer',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
 estimator.fit(train[feature_cols],train[field].to_numpy())
 return estimator

def train_models(train):
 # Benefit count and pixel damage are mathematically complementary on the same
 # changed pixels: fit benefit yield once; damage is the calibrated complement.
 # Structural harm is fitted separately and is NOT determined by benefit yield.
 return {'correction_yield':fit_regression(train,'correct_yield'),
         'structural_risk':fit_regression(train,'structural_proxy')}

def predict_outcomes(frame,models):
 b=np.clip(models['correction_yield'].predict(frame[feature_cols]),0,1)
 s=np.clip(models['structural_risk'].predict(frame[feature_cols]),0,1)
 return b,1-b,s

# True held-out-image CV on DEVELOPMENT candidates, grouping the 3 seeds together.
cvrows=[]; groups=dev.id.astype(str)
for tr,te in GroupKFold(n_splits=4).split(dev,groups=groups):
 trd,ted=dev.iloc[tr].copy(),dev.iloc[te].copy()
 assert set(trd.id.astype(str)).isdisjoint(set(ted.id.astype(str)))
 fitted=train_models(trd); pb,ph,ps=predict_outcomes(ted,fitted)
 for j,(_,r) in enumerate(ted.iterrows()):
  cvrows.append({'id':str(r.id),'seed':int(r.seed),'correct_yield':r.correct_yield,
    'pred_correct_yield':pb[j],'structural_proxy':r.structural_proxy,
    'pred_structural_proxy':ps[j], 'structural_event':r.structural_event})
cv=pd.DataFrame(cvrows)
cv.to_csv(OUT/'metrics/grouped_development_cv.csv',index=False)
models=train_models(dev)
print('Train-image grouped CV correction MAE',mean_absolute_error(cv.correct_yield,cv.pred_correct_yield))
print('Train-image grouped CV structural MAE',mean_absolute_error(cv.structural_proxy,cv.pred_structural_proxy))
if len(val):
 vb,vh,vs=predict_outcomes(val,models)
 print('Held-out development validation correction MAE',mean_absolute_error(val.correct_yield,vb))
 print('Held-out development validation structural MAE',mean_absolute_error(val.structural_proxy,vs))
# PREDECLARE coefficients, no external dataset tuning (read cfg only).
selected_margin=0.0
records['predicted_benefit'],records['predicted_harm'],records['predicted_structural_risk']=predict_outcomes(records,models)
# Expected corrected/damaged counts (different candidate size) and distinct structural harm.
records['predicted_corrected_pixels']=records.predicted_benefit*records.cost
records['predicted_damaged_pixels']=records.predicted_harm*records.cost
# Unit-cost utility facilitates fair per-pixel ranking under changing cost budgets.
records['predicted_utility']=records.predicted_benefit-cfg.utility_harm_lambda*records.predicted_harm-cfg.structural_harm_lambda*records.predicted_structural_risk
records['predicted_no_structural']=records.predicted_benefit-cfg.utility_harm_lambda*records.predicted_harm
records['predicted_benefit_only']=records.predicted_benefit
records['predicted_harm_only']=-records.predicted_harm
records['predicted_structural_only']=-records.predicted_structural_risk
records.to_csv(OUT/'candidate_tables/scored_candidates.csv',index=False)
import pickle
with open(OUT/'models/development_models.pkl','wb') as f:
 pickle.dump({'models':models,'features':feature_cols,'cfg':asdict(cfg)},f)
(OUT/'audit/selection_policy.json').write_text(json.dumps({'margin':0.,
 'lambda_pixel':cfg.utility_harm_lambda,'lambda_structural':cfg.structural_harm_lambda,
 'structural_target':'bridge fraction off GT / removed fraction on GT skeleton',
 'select_on_external_labels':False,'selected_using':'prespecified coefficients'},indent=2))
assert np.allclose(records.predicted_harm,1-records.predicted_benefit)
assert records.predicted_structural_risk.between(0,1).all()
print('Models fitted on DRIVE training candidates only. Predictions for all roles saved.')


In [ ]:
# Fair evaluation: common candidate pool, disjoint changed pixels, same base, matched actual costs.
# Compare at requested fractions of pool cost. Policies may abstain; achieved budget is explicitly reported.
strategies=['random','uncertainty','connectivity','uncertainty_connectivity','confidence','benefit_only','harm_only','structural_only','no_structural','risk_benefit','unconditional','no_repair']
def policy_order(df,policy,seed=42):
 if policy=='random':scores=np.random.default_rng(seed).random(len(df))
 elif policy=='uncertainty':scores=df.risk_mean.to_numpy()
 elif policy=='connectivity':scores=df.kind_bridge.to_numpy()*2-df.geometry.to_numpy()/max(1,cfg.max_bridge_distance)
 elif policy=='uncertainty_connectivity':scores=df.risk_mean.to_numpy()+.5*df.kind_bridge.to_numpy()-.005*df.geometry.to_numpy()
 elif policy=='confidence':scores=1-np.abs(df.prob_mean.to_numpy()-.5)*2
 elif policy=='benefit_only':scores=df.predicted_benefit_only.to_numpy()
 elif policy=='harm_only':scores=df.predicted_harm_only.to_numpy()
 elif policy=='structural_only':scores=df.predicted_structural_only.to_numpy()
 elif policy=='no_structural':scores=df.predicted_no_structural.to_numpy()
 elif policy=='risk_benefit':scores=df.predicted_utility.to_numpy()
 elif policy=='no_repair':scores=np.zeros(len(df))
 else:scores=np.arange(len(df))[::-1].astype(float)
 return np.argsort(-scores,kind='stable')

def apply_at_budget(base,candidates,df,policy,budget,randseed):
 total=int(sum(c['cost'] for c in candidates));target=int(np.floor(total*budget))
 if policy=='no_repair':return base.copy(),0,target,total,[]
 out=base.copy();used=0;accepted=[]
 for ix in policy_order(df,policy,randseed):
  c=candidates[int(df.iloc[ix].local_id)]
  if policy=='risk_benefit' and df.iloc[ix].predicted_utility<=selected_margin:continue
  if used+c['cost']>target:continue
  out.ravel()[c['idx']]=bool(c['target']);used+=c['cost'];accepted.append(c)
 return out,used,target,total,accepted

# PERFORMANCE FIX: cache GT skeleton once per image and structural outcomes once per candidate.
# Original code recomputed skeletonize(gt) for every selected removal across every
# strategy/budget. This is expensive and unnecessary; outcomes do not change.
def score_overlap_cached_gt(pred, gt, gt_skeleton):
 tp=np.sum(pred & gt); fp=np.sum(pred & ~gt); fn=np.sum(~pred & gt)
 dice=2*tp/max(1,2*tp+fp+fn)
 iou=tp/max(1,tp+fp+fn)
 sp=skeletonize(pred)
 precision=(sp & gt).sum()/max(1,sp.sum())
 recall=(gt_skeleton & pred).sum()/max(1,gt_skeleton.sum())
 cl=2*precision*recall/max(1e-9,precision+recall)
 return float(dice),float(iou),float(cl)

results=[];intervention_log=[]
print("Optimized evaluation started; per-image GT skeleton/candidate outcomes cached",flush=True)
for (name,seed,dataset,imgid),data in cache.items():
 base,gt,fov,p,risk,gate,candidates=data
 subset=records[(records.model==name)&(records.seed==seed)&(records.dataset==dataset)&(records.id==imgid)]
 if len(subset)!=len(candidates):raise RuntimeError('Candidate indexing mismatch')
 gt_eval=gt & fov
 gt_skel=skeletonize(gt_eval)
 gtflat=gt.ravel(); skflat=gt_skel.ravel()
 structural_cache={}
 outcome_cache={}
 for c in candidates:
  idx=c['idx']
  if c['type']=='bridge':proxy=float(np.mean(~gtflat[idx]))
  else:proxy=float(np.mean(skflat[idx]))
  structural_cache[c['local_id']]=(proxy,int(proxy>cfg.structural_bad_threshold))
  outcome_cache[c['local_id']]=observed_outcome(c,base,gt)
 baseline=score_overlap_cached_gt(base&fov,gt_eval,gt_skel);components0=ndlabel(base,structure=np.ones((3,3),int))[1]
 for budget in cfg.budgets:
  for policy in strategies:
   pred,used,target,total,accepted=apply_at_budget(base,candidates,subset,policy,budget,int(hashlib.sha256(f'{seed}/{dataset}/{imgid}/{budget}'.encode()).hexdigest()[:8],16))
   dice,iou,cldice=score_overlap_cached_gt(pred&fov,gt_eval,gt_skel)
   ch=(pred!=base)&fov;corr=int(np.sum(ch&(pred==gt)));harm=int(np.sum(ch&(base==gt)))
   assert corr+harm==int(ch.sum())
   ncomp=ndlabel(pred,structure=np.ones((3,3),int))[1]
   results.append({'model':name,'seed':seed,'dataset':dataset,'id':imgid,'role':next(r for r in manifest.itertuples() if r.dataset==dataset and str(r.id)==imgid).role,
       'budget':budget,'strategy':policy,'baseline_dice':baseline[0],'baseline_iou':baseline[1],'baseline_cldice':baseline[2],'dice':dice,'iou':iou,'cldice':cldice,
       'delta_dice':dice-baseline[0],'delta_cldice':cldice-baseline[2],
       'corrected':corr,'damaged':harm,'net_corrected':corr-harm,'harm_rate':harm/max(1,int(ch.sum())),
       'correction_precision':corr/max(1,int(ch.sum())),'actual_cost':used,'target_cost':target,'budget_gap':target-used,'total_pool_cost':total,
       'achieved_budget':used/max(1,total),'accepted_count':len(accepted),'abstention_rate':1-len(accepted)/max(1,len(candidates)),
       'structural_proxy_accepted':float(np.mean([structural_cache[c['local_id']][0] for c in accepted])) if accepted else 0.0,
       'structural_event_count':int(sum(structural_cache[c['local_id']][1] for c in accepted)),
       'component_delta':ncomp-components0,'component_abs_error':abs(ncomp-ndlabel(gt&fov,structure=np.ones((3,3),int))[1])})
   for c in accepted:
    co,ha=outcome_cache[c['local_id']]
    intervention_log.append({'model':name,'seed':seed,'dataset':dataset,'id':imgid,'budget':budget,'policy':policy,'local_id':c['local_id'],'cost':c['cost'],'corrected':co,'damaged':ha,'structural_proxy':structural_cache[c['local_id']][0]})
 print('Completed image',dataset,imgid,'seed',seed,'records',len(results),flush=True)
result=pd.DataFrame(results);detail=pd.DataFrame(intervention_log)
assert np.allclose(result.delta_dice,result.dice-result.baseline_dice)
assert np.allclose(result.delta_cldice,result.cldice-result.baseline_cldice)
assert (result.actual_cost<=result.target_cost).all()
assert (result.corrected+result.damaged==result.actual_cost).all()
assert (result.baseline_dice.between(0,1)).all()
assert (result[result.strategy=='no_repair'].delta_dice.abs()<1e-10).all()
assert (result[result.strategy=='no_repair'].delta_cldice.abs()<1e-10).all()
assert (result[result.strategy=='no_repair'].actual_cost==0).all()
result.to_csv(OUT/'metrics/budget_matched_per_image.csv',index=False)
detail.to_csv(OUT/'metrics/interventions_observed_for_evaluation_only.csv',index=False)
print('Per-image rows:',len(result));print(result.groupby(['dataset','strategy']).delta_dice.mean().unstack().round(4))


In [ ]:
# Evaluations and publication figures: no threshold fitting on held-out data.
held=result[result.role.isin(['test','external_test'])].copy()
held.to_csv(OUT/'metrics/heldout_only.csv',index=False)
summary=held.groupby(['dataset','budget','strategy']).agg(dice=('dice','mean'),iou=('iou','mean'),cldice=('cldice','mean'),delta_dice=('delta_dice','mean'),delta_cldice=('delta_cldice','mean'),net_corrected=('net_corrected','mean'),harm_rate=('harm_rate','mean'),structural_proxy_accepted=('structural_proxy_accepted','mean'),achieved_budget=('achieved_budget','mean'),n=('id','nunique')).reset_index()
summary.to_csv(OUT/'metrics/heldout_policy_summary.csv',index=False)
for metric,label in [('delta_dice','Dice improvement'),('net_corrected','Net corrected pixels'),('harm_rate','Fraction of changed pixels harmed'),('delta_cldice','clDice improvement'),('structural_proxy_accepted','Structural harm proxy among accepted candidates')]:
 fig,axes=plt.subplots(1,3,figsize=(16,4),sharex=True)
 for ax,(dataset,ss) in zip(axes,summary.groupby('dataset')):
  for policy in strategies:
   v=ss[ss.strategy==policy].sort_values('budget');ax.plot(v.budget,v[metric],marker='.',label=policy)
  ax.set_title(dataset);ax.set_xlabel('Requested fraction of candidate pool cost');ax.grid(alpha=.2)
 axes[0].set_ylabel(label);axes[-1].legend(fontsize=7,bbox_to_anchor=(1.03,1),loc='upper left')
 fig.tight_layout();fig.savefig(OUT/'figures'/f'{metric}_vs_budget.png',dpi=180,bbox_inches='tight');plt.close(fig)
# Paired comparisons at each budget, image-level pooled seeds per image; avoid seed pseudo-replication.
rows=[]
for dataset,df in held.groupby('dataset'):
 for budget in cfg.budgets:
  g=df[df.budget==budget]
  means=g.groupby(['id','strategy'])[['delta_dice','delta_cldice','net_corrected','harm_rate','structural_proxy_accepted']].mean().reset_index()
  for control in ['random','uncertainty','connectivity','uncertainty_connectivity','confidence','benefit_only','harm_only','structural_only','no_structural','unconditional','no_repair']:
   for metric in ['delta_dice','delta_cldice','net_corrected','harm_rate','structural_proxy_accepted']:
    wide=means.pivot(index='id',columns='strategy',values=metric)
    if 'risk_benefit' not in wide or control not in wide:continue
    d=(wide.risk_benefit-wide[control]).dropna().to_numpy()
    if not len(d):continue
    rng=np.random.default_rng(123)
    bs=np.mean(rng.choice(d,(cfg.bootstraps,len(d)),replace=True),axis=1)
    try:p=float(wilcoxon(d).pvalue) if np.any(d) else 1.
    except ValueError:p=1.
    rows.append({'dataset':dataset,'budget':budget,'control':control,'metric':metric,'n_images':len(d),'paired_mean':d.mean(),'ci_low':np.quantile(bs,.025),'ci_high':np.quantile(bs,.975),'wilcoxon_p':p})
stats=pd.DataFrame(rows)
# Benjamini-Hochberg across all tested comparisons, interpret exploratory p-values cautiously.
if len(stats):
 order=np.argsort(stats.wilcoxon_p.to_numpy());pv=stats.wilcoxon_p.to_numpy()[order];q=np.minimum.accumulate((pv*len(pv)/np.arange(1,len(pv)+1))[::-1])[::-1];adj=np.empty(len(pv));adj[order]=np.clip(q,0,1);stats['bh_fdr_q']=adj
stats.to_csv(OUT/'metrics/image_grouped_paired_bootstrap.csv',index=False)
# Budget integrated outcome (trapezoidal AUC on fixed requested budgets)
auc=[]
for (ds,pol),s in summary.groupby(['dataset','strategy']):
 s=s.sort_values('budget')
 auc.append({'dataset':ds,'strategy':pol,**{f'AUBC_{met}':float(np.trapezoid(s[met],s.budget)) for met in ['delta_dice','net_corrected','harm_rate','delta_cldice','structural_proxy_accepted']}})
pd.DataFrame(auc).to_csv(OUT/'metrics/area_under_budget_curves.csv',index=False)
print('Analysis saved to',OUT)


In [ ]:
# STAGE 4 REPRODUCIBILITY AND FAIRNESS AUDIT (held-out only)
# Examine actual achieved pixel cost. Curves on requested budgets can reward abstention
# simply because a method alters fewer pixels: compare achieved-cost coordinates too.
assert set(held.role.unique()) <= {'test','external_test'}
assert set(held.dataset.unique()) == {'DRIVE','STARE','CHASE_DB1'}
assert (held.groupby(['dataset','seed','id','budget']).strategy.nunique()==len(strategies)).all()
assert (held.actual_cost<=held.target_cost).all()
# Verify all methods share the exact same starting mask for an image/seed/budget.
shared=held.groupby(['dataset','seed','id','budget'])[['baseline_dice','baseline_cldice']].agg(['min','max'])
assert np.allclose(shared[('baseline_dice','min')],shared[('baseline_dice','max')])
assert np.allclose(shared[('baseline_cldice','min')],shared[('baseline_cldice','max')])
# Baseline should be independent of requested budget.
assert (held.groupby(['dataset','seed','id']).baseline_dice.nunique()==1).all()
# A fair caution: do not call a requested-budget comparison 'equal achieved cost'
# when policies abstain. Report achieved costs and gaps explicitly.
fair=held.groupby(['dataset','budget','strategy']).agg(
 mean_actual_cost=('actual_cost','mean'),mean_target_cost=('target_cost','mean'),
 mean_budget_gap=('budget_gap','mean'),mean_achieved_fraction=('achieved_budget','mean'),
 delta_dice=('delta_dice','mean'),delta_cldice=('delta_cldice','mean'),
 net_corrected=('net_corrected','mean'),harm_rate=('harm_rate','mean'),structural_proxy_accepted=('structural_proxy_accepted','mean')).reset_index()
fair.to_csv(OUT/'metrics/achieved_cost_fairness_audit.csv',index=False)
# A descriptive achieved-cost frontier: no interpolation extrapolation beyond observed support.
curves=[]
for (dataset,policy),part in fair.groupby(['dataset','strategy']):
 s=part.sort_values('mean_actual_cost')
 curves.append({'dataset':dataset,'strategy':policy,
  'cost_min':float(s.mean_actual_cost.min()),'cost_max':float(s.mean_actual_cost.max()),
  'AUC_net_corrected_vs_achieved_cost':float(np.trapezoid(s.net_corrected,s.mean_actual_cost)),
  'AUC_delta_dice_vs_achieved_cost':float(np.trapezoid(s.delta_dice,s.mean_actual_cost))})
pd.DataFrame(curves).to_csv(OUT/'metrics/achieved_cost_descriptive_integrals.csv',index=False)
# Save a diagnostic plot on actual cost; it MUST NOT be presented as mathematically
# exact matched-cost comparison because candidate modifications are discrete.
for dataset,part in fair.groupby('dataset'):
 plt.figure(figsize=(8,5))
 for policy,s in part.groupby('strategy'):
  s=s.sort_values('mean_actual_cost')
  plt.plot(s.mean_actual_cost,s.delta_dice,marker='o',label=policy)
 plt.axhline(0,ls='--',c='gray',lw=1)
 plt.xlabel('Achieved changed pixels (dataset mean)');plt.ylabel('Delta Dice vs identical starting mask')
 plt.title(dataset+': achieved-cost diagnostic');plt.legend(fontsize=7)
 plt.tight_layout();plt.savefig(OUT/'figures'/f'{dataset}_achieved_cost_diagnostic.png',dpi=180);plt.close()
print('Corrected baseline and achieved-cost validation checks passed')
print(fair[fair.budget==.2].round(4).to_string(index=False))


In [ ]:
# Qualitative genuine mask overlays: explicitly 2D vessels and 3D height plots of 2D confidence ONLY.
from mpl_toolkits.mplot3d import Axes3D
for dataset in ['DRIVE','STARE','CHASE_DB1']:
 matches=[k for k in cache if k[2]==dataset and k[1]==cfg.seeds[0]]
 if not matches:continue
 key=matches[0];base,gt,fov,p,risk,gate,candidates=cache[key]
 name,seed,ds,imgid=key;df=records[(records.model==name)&(records.seed==seed)&(records.dataset==ds)&(records.id==imgid)]
 pred,_,_,_,_=apply_at_budget(base,candidates,df,'risk_benefit',.4,42)
 fig,axs=plt.subplots(1,4,figsize=(15,4))
 for ax,v,t in zip(axs,[gt,base,pred,(pred!=gt)&fov],['Ground truth','Frozen v9 base','Benefit-selected repair','Error after repair']):ax.imshow(v,cmap='gray');ax.set_title(t);ax.axis('off')
 fig.tight_layout();fig.savefig(OUT/'figures'/f'{ds}_qualitative.png',dpi=180);plt.close(fig)
 # A height-field visualization is not an OCT/3D retinal volume.
 step=max(2,max(p.shape)//140);yy,xx=np.mgrid[0:p.shape[0]:step,0:p.shape[1]:step]
 fig=plt.figure(figsize=(8,5));ax=fig.add_subplot(111,projection='3d');ax.plot_surface(xx,yy,p[::step,::step],cmap='viridis',linewidth=0);ax.set_title(f'{ds}: 2D vessel probability represented as a 3D surface');ax.set_zlim(0,1)
 fig.savefig(OUT/'figures'/f'{ds}_probability_surface_NOT_3D_ANATOMY.png',dpi=160);plt.close(fig)


In [ ]:
# Stage 5 final scientific sanity / validity gates and image-seed coverage.
# This is descriptive; not a certification of real Kaggle execution.
held = result[result.role.isin(['test','external_test'])].copy()
assert held.dataset.nunique()==3
assert all(held.groupby(['dataset','id','seed','budget']).strategy.nunique()==len(strategies))
assert (held.actual_cost<=held.target_cost).all()
assert held.groupby(['dataset','id','seed']).baseline_dice.nunique().max()==1
assert held[held.strategy=='no_repair'].delta_dice.abs().max()<1e-10
assert held[held.strategy=='no_repair'].delta_cldice.abs().max()<1e-10
# No fabricated or interpolated budget-equivalence claims.
cost_spread=held.groupby(['dataset','budget']).achieved_budget.agg(['min','max'])
cost_spread['difference']=cost_spread['max']-cost_spread['min']
cost_spread.to_csv(OUT/'audit/achieved_budget_spread.csv')
# Non-complementarity check: both benefit and structural targets must actually vary.
assert cv.structural_proxy.nunique()>1, 'No variation in structural target: cannot claim structural-risk model.'
assert cv.correct_yield.nunique()>1, 'No variation in benefit target.'
# Repeated seeds share the same image IDs: statistical tests above average within image.
print('STAGE 5 VALIDATION CHECKS PASSED')
print('Unique held-out images',held.groupby('dataset').id.nunique().to_dict())
print('Maximum achieved-budget spread across strategies',float(cost_spread.difference.max()))
print('Interpretation: cost disparities must be disclosed; Stage 5 is NOT automatically exact-cost matched.')


In [ ]:
# MATCHES STAGE 5 POLICY ORDER AND SELECTION (original Stage5 code, not approximated)
VIS_BUDGET = 0.20
VIS_SEED = int(cfg.seeds[0]) # one representative frozen checkpoint for qualitative illustrations
MAX_FIGURES_PER_DATASET = 2  # fixed first case plus policy-disagreement example (GT-free selection)

def policy_order_visual(df, policy, seed=42):
    if policy == 'benefit_only': scores=df.predicted_benefit_only.to_numpy()
    elif policy == 'risk_benefit': scores=df.predicted_utility.to_numpy()
    else: raise ValueError('Visualization restricted to Stage5 benefit_only and risk_benefit')
    return np.argsort(-scores, kind='stable')

def apply_visual(base, candidates, df, policy, budget=VIS_BUDGET):
    assert len(candidates)==len(df)
    assert sorted(df.local_id.astype(int).tolist())==list(range(len(candidates)))
    total=int(sum(c['cost'] for c in candidates))
    target=int(np.floor(total*budget))
    out=base.copy(); used=0; accepted=[]
    for row_index in policy_order_visual(df,policy):
        row=df.iloc[int(row_index)]
        c=candidates[int(row.local_id)]
        if policy=='risk_benefit' and row.predicted_utility<=0.0:continue
        if used+c['cost']>target:continue
        out.ravel()[c['idx']]=bool(c['target'])
        used+=c['cost'];accepted.append(int(c['local_id']))
    assert used<=target
    return out, {'accepted_ids':accepted,'cost_used':used,'cost_target':target,'total_cost':total}

# Unit checks: empty-candidate case and actual-budget upper bound.
_b=np.zeros((10,10),bool)
_empty=pd.DataFrame(columns=['local_id','predicted_benefit_only','predicted_utility'])
_o,_a=apply_visual(_b,[],_empty,'risk_benefit')
assert np.array_equal(_b,_o) and _a['cost_used']==0
print('Stage5 visualization policy tests passed')

In [ ]:
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from scipy.ndimage import binary_dilation as nd_dilate
from pathlib import Path

visual_root=OUT/'figures'/'Six_Panel'
visual_root.mkdir(parents=True,exist_ok=True)
figure_records=[]; visual_cache={}

for ds in ('DRIVE','STARE','CHASE_DB1'):
    keys=[key for key in cache if key[2]==ds and key[1]==VIS_SEED]
    keys=[key for key in keys if (ds!='DRIVE' or next(r.role for r in manifest.itertuples() if r.dataset==ds and str(r.id)==key[3])=='test')]
    keys=sorted(keys,key=lambda x:x[3]); assert keys, f'No held-out images for {ds}'
    ranked=[]
    for key in keys:
        name,seed,dataset,imgid=key
        base,gt,fov,p,risk,gate,candidates=cache[key]
        sub=records[(records.model==name)&(records.seed==seed)&(records.dataset==dataset)&(records.id==imgid)].sort_values('local_id')
        benefit,bs=apply_visual(base,candidates,sub,'benefit_only')
        riskpred,rs=apply_visual(base,candidates,sub,'risk_benefit')
        difference=int(np.sum(benefit!=riskpred))
        visual_cache[key]=(benefit,riskpred,bs,rs,difference)
        ranked.append((difference,key))
    selected=[keys[0]]
    # Include the manuscript's CHASE example when present.
    if ds == 'CHASE_DB1':
        chosen = [k for k in keys if str(k[3]).lower().replace('_','') in ('image08l','08l')]
        if chosen and chosen[0] not in selected: selected.append(chosen[0])
    most_different=max(ranked,key=lambda z:(z[0],z[1][3]))[1]
    if most_different!=keys[0] and MAX_FIGURES_PER_DATASET>=2:selected.append(most_different)
    for case,key in enumerate(selected,1):
        name,seed,dataset,imgid=key
        base,gt,fov,p,risk,gate,candidates=cache[key]
        benefit,riskpred,bs,rs,difference=visual_cache[key]
        row=manifest[(manifest.dataset==dataset)&(manifest.id.astype(str)==str(imgid))]
        assert len(row)==1
        rgb=open_rgb(row.iloc[0]['image'])
        assert rgb.shape[:2]==base.shape==gt.shape==fov.shape
        # Stage5 decisions made before consulting GT; only here generate explanatory error colors.
        changed=(riskpred!=base)&fov
        correct=changed&(riskpred==gt)
        damaged=changed&(riskpred!=gt)
        assert (correct|damaged).sum()==changed.sum()
        only_benefit=((benefit!=base)&(riskpred==base))&fov
        overlay=np.zeros((*gt.shape,3),dtype=float)
        # Dim original RGB underneath so structural changes remain visible
        overlay[:]=rgb/255*0.30
        overlay[correct]=[0.0,0.95,0.26]  # corrected
        overlay[damaged]=[1.0,0.16,0.15]  # damaged
        outline=nd_dilate(only_benefit,iterations=2)&(~only_benefit)
        overlay[outline]=[0.05,0.86,1.0]
        fig,axs=plt.subplots(2,3,figsize=(15,10),layout='constrained')
        ims=[rgb,gt,base,benefit,riskpred,overlay]
        titles=['A  Original retinal image','B  Ground-truth vessel mask','C  Frozen v9 base',
                'D  Benefit-only repair','E  Benefit–harm-aware repair','F  Decisions and observed errors']
        for ax,im,title in zip(axs.flat,ims,titles):
            if im.ndim==2:ax.imshow(im,cmap='gray',vmin=0,vmax=1,interpolation='nearest')
            else:ax.imshow(im,interpolation='nearest')
            ax.set_title(title,fontsize=12);ax.set_axis_off()
        axs[1,2].legend(handles=[Patch(color='#00f242',label='Corrected pixels'),Patch(color='#ff2926',label='Newly damaged pixels'),Patch(color='#0cdbff',label='Benefit-only changes not adopted (outline)')],loc='lower center',fontsize=8,framealpha=.85)
        fig.suptitle(f'{dataset} | ID {imgid} | selective_guard seed {seed} | requested budget 20% | case {case}',fontsize=14)
        suffix=f'{ds}_{imgid}_seed{seed}_case{case}'.replace('/','_')
        fig.savefig(visual_root/f'{suffix}_SIX_PANEL.png',dpi=250,bbox_inches='tight')
        fig.savefig(visual_root/f'{suffix}_SIX_PANEL.pdf',bbox_inches='tight')
        plt.close(fig)
        # Zoom crop centered on actual modification disagreement, fallback to risk-policy changed pixels.
        diff=(benefit!=riskpred)&fov
        active=np.argwhere(diff if np.any(diff) else changed)
        if not len(active):active=np.argwhere(gt)
        cy,cx=active[len(active)//2] if len(active) else (gt.shape[0]//2,gt.shape[1]//2)
        half=85;y0=max(0,int(cy)-half);y1=min(gt.shape[0],int(cy)+half)
        x0=max(0,int(cx)-half);x1=min(gt.shape[1],int(cx)+half)
        fig,zaxs=plt.subplots(1,4,figsize=(16,4),layout='constrained')
        for ax,arr,title in zip(zaxs,[base,benefit,riskpred,overlay],['Frozen base','Benefit-only','Benefit–harm-aware','Decision/error overlay']):
            crop=arr[y0:y1,x0:x1]
            ax.imshow(crop,cmap='gray',vmin=0,vmax=1,interpolation='nearest') if crop.ndim==2 else ax.imshow(crop,interpolation='nearest')
            ax.set_title(title);ax.axis('off')
        fig.suptitle(f'{dataset} {imgid}: registered crop ({x0}:{x1}, {y0}:{y1})')
        fig.savefig(visual_root/f'{suffix}_ZOOM.png',dpi=250,bbox_inches='tight')
        fig.savefig(visual_root/f'{suffix}_ZOOM.pdf',bbox_inches='tight');plt.close(fig)
        # Record metrics, under GT only for final audit; no policy selection from these values.
        base_eval=score_overlap(base&fov,gt&fov)
        benefit_eval=score_overlap(benefit&fov,gt&fov)
        risk_eval=score_overlap(riskpred&fov,gt&fov)
        figure_records.append(dict(dataset=ds,id=imgid,seed=seed,case=case,selection='first-ID' if key==keys[0] else 'maximum-policy-disagreement-GT-free',
            candidate_count=len(candidates),policy_disagreement_pixels=difference,
            benefit_accepted=len(bs['accepted_ids']),risk_accepted=len(rs['accepted_ids']),benefit_cost=bs['cost_used'],risk_cost=rs['cost_used'],target_cost=rs['cost_target'],
            base_dice=base_eval[0],benefit_dice=benefit_eval[0],risk_benefit_dice=risk_eval[0],
            base_cldice=base_eval[2],benefit_cldice=benefit_eval[2],risk_benefit_cldice=risk_eval[2],
            risk_corrected_pixels=int(correct.sum()),risk_damaged_pixels=int(damaged.sum()),figure=f'{suffix}_SIX_PANEL.png'))
        print('Exported',suffix,'disagreement',difference,'risk repairs',len(rs['accepted_ids']),flush=True)
fig_index=pd.DataFrame(figure_records)
fig_index.to_csv(OUT/'audit'/'SIX_PANEL_FIGURE_INDEX.csv',index=False)
print(fig_index[['dataset','id','selection','policy_disagreement_pixels','benefit_dice','risk_benefit_dice','figure']].to_string(index=False))

In [ ]:
# Export manuscript-focused, machine-readable tables from the computed results.
report_dir=OUT/'manuscript_tables'
report_dir.mkdir(exist_ok=True)
summary.to_csv(report_dir/'table_policy_summary_all_budgets.csv',index=False)
summary[np.isclose(summary.budget,.2)].to_csv(report_dir/'table_policy_summary_20_percent.csv',index=False)
fair[np.isclose(fair.budget,.2)].to_csv(report_dir/'table_budget_utilization_20_percent.csv',index=False)
stats[(stats.control=='benefit_only') & np.isclose(stats.budget,.2)].to_csv(report_dir/'table_paired_risk_vs_benefit_20_percent.csv',index=False)
pd.DataFrame(auc).to_csv(report_dir/'table_area_under_budget_curves.csv',index=False)
held.groupby(['dataset','id','seed'])[['baseline_dice','baseline_cldice']].first().groupby('dataset').agg(['mean','std','count']).to_csv(report_dir/'table_frozen_baselines.csv')
# Hash actual inputs so comparisons can identify different checkpoint artifacts.
def sha256_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for block in iter(lambda:f.read(1024*1024), b''):h.update(block)
 return h.hexdigest()
input_hashes=[{'model':k[0],'seed':k[1],'file':str(v[1]),'sha256':sha256_file(v[1])} for k,v in registry.items()]
input_hashes += [{'model':'locked_manifest','seed':None,'file':str(manifest_files[0]),'sha256':sha256_file(manifest_files[0])}, {'model':'threshold_table','seed':None,'file':str(sorted(set(sel_files))[0]),'sha256':sha256_file(sorted(set(sel_files))[0])}]
pd.DataFrame(input_hashes).to_csv(OUT/'audit/input_sha256.csv',index=False)
import platform
(OUT/'audit/runtime_versions.json').write_text(json.dumps({'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'torch':torch.__version__,'opencv':cv2.__version__},indent=2))
print('Manuscript-focused tables and input hashes saved.')


In [ ]:
# Final research audit, package outputs, and reproducibility report.
report=['VASC-GUARD STAGE5 — CHECKPOINT-BASED RISK-BENEFIT INTERVENTION','',
 'No segmentation model retraining; original v9 architectures loaded strict=True.',
 'New candidate generator is NOT identical to original learned residual repair.',
 'Separate correction-yield and structural-risk estimators fitted only on DRIVE training; development predictions are in-sample, NOT OOF.',
 'Corrected Dice baseline uses the unmodified raw starting prediction, not base intersect GT.',
 'Actual achieved intervention costs differ due to abstention; achieved-cost diagnostic is descriptive, not exactly cost-matched.',
 'Abstention margin fixed at zero; DRIVE test/STARE/CHASE labels not used for policy fitting.',
 'External study is exploratory because researchers previously inspected external dataset results.',
 'Matched costs measured in changed pixels, with abstention and discrete budget deviations reported.',
 'Topology component differences are coarse proxy measures, not proven false bridge rates.',
 '3D figures are visualizations of 2D probability maps, NOT volumetric imaging.',
 f'Images evaluated: {counts.groupby("dataset").id.nunique().to_dict()}',
 f'Candidates: {len(records)}',
 f'Model checkpoints: {len(registry)}',
 f'Estimated selection margin: {selected_margin}',
 'Pixel benefit and pixel harm fractions are complementary; structural harm uses an additional proxy target.',
 'Structural harm proxy measures unsupported additions or GT-skeleton removal, not a validated clinical anatomy endpoint.',
 'No superior performance may be claimed without inspecting output comparisons.']
(OUT/'reports/RESEARCH_LIMITATIONS_AND_PROTOCOL.txt').write_text('\n'.join(report))
(OUT/'audit/config.json').write_text(json.dumps(asdict(cfg),indent=2))
package=Path('/kaggle/working/VASC_Guard_Manuscript_Complete_Results.zip')
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
 for p in OUT.rglob('*'):
  if p.is_file() and p.resolve()!=package.resolve():z.write(p,p.relative_to(OUT.parent))
print('RESULT ZIP:',package,'MB:',round(package.stat().st_size/1e6,2))
print('\n'.join(report))
